<h1 style="background-color: #d0ebff; color: #1a1a1a; display: inline-block; padding: 10px 18px; border-radius: 10px; font-size: 30px;">
Extensión: detector de autos subvaluados
</h1>

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Proceso de diseño de la extensión
</h3>

Antes de implementar la extensión, pensamos qué tipo de aporte podía sumar más valor al trabajo. En lugar de agregar solo un gráfico descriptivo, elegimos construir una herramienta basada en el modelo: un detector de publicaciones potencialmente subvaluadas. La idea es usar el precio esperado por el modelo como referencia y compararlo contra el precio publicado. Si el precio publicado está bastante por debajo del esperado, esa publicación puede ser una oportunidad interesante para revisar con más detalle.

El proceso se organiza en cuatro pasos:

1. **Estimar un precio esperado para cada auto.** Para eso entrenamos modelos de XGBoost usando las variables disponibles después del preprocesamiento. Como el precio tiene una distribución muy asimétrica, se entrena sobre `log(Precio)` y luego se vuelve a la escala original.

2. **Evitar evaluar una fila con un modelo que ya la vio.** Para que el ranking sea más confiable, usamos predicciones out-of-fold: se divide el dataset en folds y cada publicación es predicha por un modelo entrenado con las demás filas. Esto reduce el riesgo de que el modelo simplemente memorice casos del entrenamiento.

3. **Comparar precio esperado y precio publicado.** Para cada publicación calculamos la diferencia absoluta y porcentual entre ambos valores. Una diferencia positiva grande indica que el modelo esperaba un precio mayor que el observado.

4. **Convertir el resultado en una herramienta de análisis.** Además del ranking general, dejamos funciones para filtrar por marca o modelo. La idea es que una persona interesada en comprar un vehículo pueda buscar oportunidades dentro de un segmento específico.

Este enfoque no garantiza que un auto sea realmente una buena compra, porque el dataset no incluye toda la información relevante, como estado mecánico real, ubicación, urgencia del vendedor, deudas o detalles legales. Por eso, interpretamos el resultado como un primer filtro para detectar publicaciones que merecen una revisión manual más cuidadosa.


En este notebook se desarrolla un primer borrador de una aplicación útil para compra y venta de autos: detectar publicaciones cuyo precio observado parece estar por debajo de su valor esperado.

De las extensiones propuestas por la cátedra, elegimos esta porque conecta directamente el modelo predictivo con una decisión práctica: encontrar oportunidades de compra. La idea es estimar un **precio esperado** para cada vehículo y comparar ese valor con el precio publicado.

Para hacerlo de forma académicamente defendible, no usamos predicciones in-sample. En cambio, usamos predicciones **out-of-fold**: cada auto es evaluado por un modelo que fue entrenado sin incluir esa observación. Esto reduce el riesgo de leakage y evita que el ranking de oportunidades esté basado en memorizar el dataset.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
1. Imports and configuration
</h3>

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

sys.path.append("../src")
import constants as const

RANDOM_STATE = 42
TARGET = "Precio"
DATA_PATH = Path("../data/processed/dataset_pp.csv")

PREMIUM_BRANDS = const.PREMIUM_BRANDS
TEXT_COLS_TO_DROP = ["Título", "Descripción", "Versión"]
NOISY_COLS_TO_DROP = ["Color", "Con cámara de retroceso"]

XGB_DEAL_FINDER_PARAMS = {
    "n_estimators": 350,
    "learning_rate": 0.04,
    "max_depth": 5,
    "min_child_weight": 4,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.10,
    "reg_lambda": 3.00,
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
2. Load data and remove duplicated listings
</h3>

Partimos de `dataset_pp.csv`, que ya contiene el dataset preprocesado: precios unificados en USD, variables limpias y features de texto ya extraídas.

Antes de modelar eliminamos duplicados exactos. Esto es importante porque si una misma publicación aparece más de una vez, el modelo podría entrenarse con una copia y evaluar otra copia casi idéntica, generando leakage o una evaluación demasiado optimista.

In [ ]:
dataset_pp = pd.read_csv(DATA_PATH)
dataset = dataset_pp.drop_duplicates().reset_index(drop=True)

pd.DataFrame([
    {
        "stage": "before_deduplication",
        "rows": len(dataset_pp),
        "exact_duplicates": dataset_pp.duplicated().sum(),
    },
    {
        "stage": "after_deduplication",
        "rows": len(dataset),
        "exact_duplicates": dataset.duplicated().sum(),
    },
])


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
3. Feature preparation functions
</h3>

En esta primera versión usamos una representación simple y estable:

- `Marca`, `Modelo` y `Marca_Modelo` se codifican con label encoding aprendido solo con el fold de entrenamiento.
- `Marca_Modelo` captura la interacción entre marca y modelo, que suele ser muy informativa para precio.
- `Es_Alta_Gama` marca si la marca pertenece a un grupo premium.
- Las variables categóricas restantes se codifican con one-hot encoding.
- `Color` y `Con cámara de retroceso` se eliminan en este borrador porque en los experimentos previos parecían aportar poco o agregar ruido.

El punto importante es que cada transformación se aprende solamente con los datos de entrenamiento de cada fold y luego se aplica al fold de validación.

In [ ]:
def add_brand_model_feature(train_df, test_df, min_count=20):
    """
    Adds a brand-model interaction using frequencies learned from train only.

    Rare brand-model combinations are grouped as 'other'. This avoids creating
    unstable categories with very few observations.
    """
    train_data = train_df.copy()
    test_data = test_df.copy()

    train_combo = train_data["Marca"].astype(str) + "_" + train_data["Modelo"].astype(str)
    test_combo = test_data["Marca"].astype(str) + "_" + test_data["Modelo"].astype(str)

    frequent_combos = train_combo.value_counts()
    frequent_combos = frequent_combos[frequent_combos >= min_count].index

    train_data["Marca_Modelo"] = train_combo.where(train_combo.isin(frequent_combos), "other")
    test_data["Marca_Modelo"] = test_combo.where(test_combo.isin(frequent_combos), "other")

    return train_data, test_data


def add_premium_brand_feature(df, brand_col="Marca", premium_brands=PREMIUM_BRANDS):
    """
    Adds a binary feature that identifies high-end brands.
    """
    data = df.copy()
    data["Es_Alta_Gama"] = data[brand_col].isin(premium_brands).astype(int)
    return data


def fit_label_encoding(train_values):
    """
    Learns a label-encoding mapping from train values only.
    """
    categories = sorted(train_values.fillna("missing").astype(str).unique())
    return {category: code for code, category in enumerate(categories)}


def apply_label_encoding(values, mapping, unknown_value=-1):
    """
    Applies a train-learned label mapping. Unknown categories are coded as -1.
    """
    return values.fillna("missing").astype(str).map(mapping).fillna(unknown_value).astype(int)


def group_rare_categories(train_df, test_df, columns, min_count=20, rare_label="otros"):
    """
    Groups rare categorical values using train frequencies only.
    """
    train_data = train_df.copy()
    test_data = test_df.copy()

    for column in columns:
        if column not in train_data.columns or column not in test_data.columns:
            continue

        train_values = train_data[column].fillna("missing").astype(str)
        test_values = test_data[column].fillna("missing").astype(str)

        counts = train_values.value_counts()
        frequent_values = counts[counts >= min_count].index

        train_data[column] = train_values.where(train_values.isin(frequent_values), rare_label)
        test_data[column] = test_values.where(test_values.isin(frequent_values), rare_label)

    return train_data, test_data


def build_model_features(train_df, test_df):
    """
    Builds leakage-safe model matrices for one train/test fold.

    The function receives raw features for a fold, learns encoders from train,
    and applies the same transformations to test.
    """
    train_features, test_features = add_brand_model_feature(train_df, test_df, min_count=20)

    train_features = add_premium_brand_feature(train_features)
    test_features = add_premium_brand_feature(test_features)

    columns_to_drop = TEXT_COLS_TO_DROP + NOISY_COLS_TO_DROP
    train_features = train_features.drop(columns=columns_to_drop, errors="ignore")
    test_features = test_features.drop(columns=columns_to_drop, errors="ignore")

    label_cols = ["Marca", "Modelo", "Marca_Modelo"]

    for column in label_cols:
        if column in train_features.columns and column in test_features.columns:
            mapping = fit_label_encoding(train_features[column])
            train_features[column] = apply_label_encoding(train_features[column], mapping)
            test_features[column] = apply_label_encoding(test_features[column], mapping)

    object_cols = train_features.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    train_features, test_features = group_rare_categories(
        train_features,
        test_features,
        columns=object_cols,
        min_count=20,
    )

    train_encoded = pd.get_dummies(train_features, columns=object_cols, dtype=int)
    test_encoded = pd.get_dummies(test_features, columns=object_cols, dtype=int)

    test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)

    return train_encoded, test_encoded


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
4. Out-of-fold fair price estimation
</h3>

Para detectar autos subvaluados necesitamos estimar un precio esperado. Si entrenáramos un modelo con todo el dataset y luego predijéramos sobre ese mismo dataset, el resultado podría estar contaminado por memorización.

Por eso usamos predicciones **out-of-fold**:

1. dividimos el dataset en folds;
2. entrenamos el modelo con todos los folds menos uno;
3. predecimos el fold que quedó afuera;
4. repetimos hasta que cada auto tenga una predicción hecha por un modelo que no lo vio durante el entrenamiento.

Esta predicción out-of-fold se interpreta como una estimación más honesta del valor esperado del vehículo.

In [ ]:
def build_xgboost_model(params=XGB_DEAL_FINDER_PARAMS):
    """
    Builds the XGBoost pipeline used for fair-price estimation.
    """
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("regressor", XGBRegressor(**params)),
    ])


def compute_oof_predictions(data, target_col=TARGET, n_splits=5, random_state=RANDOM_STATE):
    """
    Computes out-of-fold predictions for every row in the dataset.

    The target is modeled in log scale and transformed back to USD for
    interpretation.
    """
    X = data.drop(columns=[target_col])
    y = data[target_col]

    oof_predictions = pd.Series(index=data.index, dtype=float)
    fold_metrics = []

    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (train_idx, test_idx) in enumerate(kfold.split(X), start=1):
        X_train_raw = X.iloc[train_idx].copy()
        X_test_raw = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        X_train, X_test = build_model_features(X_train_raw, X_test_raw)

        model = build_xgboost_model()
        model.fit(X_train, np.log1p(y_train))

        fold_predictions = np.expm1(model.predict(X_test))
        fold_predictions = np.maximum(fold_predictions, 0)
        oof_predictions.iloc[test_idx] = fold_predictions

        fold_metrics.append({
            "fold": fold,
            "rmse": root_mean_squared_error(y_test, fold_predictions),
            "mae": mean_absolute_error(y_test, fold_predictions),
            "r2": r2_score(y_test, fold_predictions),
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_features": X_train.shape[1],
        })

    return oof_predictions, pd.DataFrame(fold_metrics)


In [ ]:
oof_predicted_price, fold_metrics = compute_oof_predictions(dataset, n_splits=5)

fold_metrics.style.hide(axis="index").format({
    "rmse": "{:,.2f}",
    "mae": "{:,.2f}",
    "r2": "{:.4f}",
})


In [ ]:
overall_metrics = pd.DataFrame([
    {
        "rmse": root_mean_squared_error(dataset[TARGET], oof_predicted_price),
        "mae": mean_absolute_error(dataset[TARGET], oof_predicted_price),
        "r2": r2_score(dataset[TARGET], oof_predicted_price),
    }
])

overall_metrics.style.hide(axis="index").format({
    "rmse": "{:,.2f}",
    "mae": "{:,.2f}",
    "r2": "{:.4f}",
})


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
5. Deal scoring: how undervalued is each car?
</h3>

Con el precio esperado estimado, definimos dos medidas:

- `discount_vs_expected_pct`: cuánto menor es el precio publicado respecto del precio esperado, medido sobre el precio esperado.
- `resale_gain_pct`: ganancia potencial relativa al precio de compra, es decir, cuánto podría aumentar el valor si se vendiera al precio esperado.

Para ranking de oportunidades, ordenamos por `resale_gain_pct`, porque representa la ganancia relativa para quien compra.

In [ ]:
def build_deal_table(data, predicted_price, target_col=TARGET):
    """
    Builds a table of potential deals from observed and predicted prices.
    """
    deals = data.copy()
    deals["expected_price"] = predicted_price.values
    deals["listed_price"] = deals[target_col]
    deals["absolute_gap"] = deals["expected_price"] - deals["listed_price"]

    deals["discount_vs_expected_pct"] = np.where(
        deals["expected_price"] > 0,
        deals["absolute_gap"] / deals["expected_price"] * 100,
        np.nan,
    )

    deals["resale_gain_pct"] = np.where(
        deals["listed_price"] > 0,
        deals["absolute_gap"] / deals["listed_price"] * 100,
        np.nan,
    )

    deals["is_undervalued"] = deals["absolute_gap"] > 0

    return deals.sort_values("resale_gain_pct", ascending=False).reset_index(drop=True)


deal_table = build_deal_table(dataset, oof_predicted_price)
plausible_deals = deal_table[deal_table["listed_price"] >= 10_000].copy()

plausible_deals[
    [
        "Marca", "Modelo", "Versión", "Año", "Kilómetros",
        "listed_price", "expected_price", "absolute_gap",
        "discount_vs_expected_pct", "resale_gain_pct",
    ]
].head(10).style.hide(axis="index").format({
    "listed_price": "{:,.2f}",
    "expected_price": "{:,.2f}",
    "absolute_gap": "{:,.2f}",
    "discount_vs_expected_pct": "{:.2f}%",
    "resale_gain_pct": "{:.2f}%",
})


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
6. User-facing search function
</h3>

La siguiente función funciona como una aplicación sencilla: dada una marca y/o modelo, devuelve las publicaciones que parecen estar subvaluadas por encima de un umbral.

El umbral recomendado para empezar es 15% o 20%. Un umbral muy bajo puede devolver autos que simplemente están dentro del error normal del modelo; un umbral alto devuelve menos casos, pero más interesantes para inspección manual. Además se usa un precio mínimo publicado para evitar oportunidades aparentes que probablemente sean errores de carga.

In [ ]:
def normalize_filter_text(value):
    """
    Normalizes user text filters for case-insensitive matching.
    """
    if value is None:
        return None
    return str(value).strip().lower()


def find_undervalued_listings(
    deals,
    brand=None,
    model=None,
    min_discount_pct=15,
    min_expected_price=10_000,
    min_listed_price=10_000,
    top_n=20,
):
    """
    Returns listings whose observed price is meaningfully below expected price.

    Arguments:
        deals (pd.DataFrame): output from build_deal_table
        brand (str | None): optional brand filter
        model (str | None): optional model filter
        min_discount_pct (float): minimum discount versus expected price
        min_expected_price (float): lower bound to avoid unstable expected values
        min_listed_price (float): lower bound to avoid likely price-entry errors
        top_n (int): number of results to show

    Returns:
        pd.DataFrame: ranked undervalued listings
    """
    result = deals.copy()

    brand = normalize_filter_text(brand)
    model = normalize_filter_text(model)

    if brand is not None:
        result = result[result["Marca"].astype(str).str.lower() == brand]

    if model is not None:
        result = result[result["Modelo"].astype(str).str.lower() == model]

    result = result[
        (result["discount_vs_expected_pct"] >= min_discount_pct)
        & (result["expected_price"] >= min_expected_price)
        & (result["listed_price"] >= min_listed_price)
    ]

    output_cols = [
        "Marca", "Modelo", "Versión", "Año", "Kilómetros", "Tipo de vendedor",
        "listed_price", "expected_price", "absolute_gap",
        "discount_vs_expected_pct", "resale_gain_pct",
    ]

    return (
        result[output_cols]
        .sort_values("resale_gain_pct", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


In [ ]:
# Example 1: global top opportunities
find_undervalued_listings(
    deal_table,
    min_discount_pct=20,
    top_n=10,
).style.hide(axis="index").format({
    "listed_price": "{:,.2f}",
    "expected_price": "{:,.2f}",
    "absolute_gap": "{:,.2f}",
    "discount_vs_expected_pct": "{:.2f}%",
    "resale_gain_pct": "{:.2f}%",
})


In [ ]:
# Example 2: change these filters to inspect a specific market segment
find_undervalued_listings(
    deal_table,
    brand="volkswagen",
    model=None,
    min_discount_pct=15,
    top_n=10,
).style.hide(axis="index").format({
    "listed_price": "{:,.2f}",
    "expected_price": "{:,.2f}",
    "absolute_gap": "{:,.2f}",
    "discount_vs_expected_pct": "{:.2f}%",
    "resale_gain_pct": "{:.2f}%",
})


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
7. Which brand-model groups show more opportunities?
</h3>

Además de mirar autos individuales, podemos resumir por marca y modelo. Esto ayuda a detectar segmentos donde aparecen más oportunidades subvaluadas o donde el modelo tiende a encontrar mayores diferencias entre precio observado y esperado.

In [ ]:
undervalued_summary = (
    deal_table[(deal_table["discount_vs_expected_pct"] >= 15) & (deal_table["listed_price"] >= 10_000)]
    .groupby(["Marca", "Modelo"])
    .agg(
        opportunities=("listed_price", "size"),
        median_discount_pct=("discount_vs_expected_pct", "median"),
        max_resale_gain_pct=("resale_gain_pct", "max"),
        median_listed_price=("listed_price", "median"),
        median_expected_price=("expected_price", "median"),
    )
    .reset_index()
    .sort_values(["opportunities", "max_resale_gain_pct"], ascending=False)
)

undervalued_summary.head(20).style.hide(axis="index").format({
    "median_discount_pct": "{:.2f}%",
    "max_resale_gain_pct": "{:.2f}%",
    "median_listed_price": "{:,.2f}",
    "median_expected_price": "{:,.2f}",
})


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
8. Visualization
</h3>

Este gráfico compara el precio publicado contra el precio esperado. Los puntos por encima de la diagonal son casos donde el precio esperado es mayor al precio publicado, es decir, posibles autos subvaluados.

In [ ]:
def plot_expected_vs_listed(deals, min_discount_pct=20, sample_size=4000):
    """
    Plots listed price against expected price and highlights undervalued cars.
    """
    plot_data = deals.copy()

    if len(plot_data) > sample_size:
        plot_data = plot_data.sample(sample_size, random_state=RANDOM_STATE)

    is_opportunity = plot_data["discount_vs_expected_pct"] >= min_discount_pct

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.scatter(
        plot_data.loc[~is_opportunity, "listed_price"],
        plot_data.loc[~is_opportunity, "expected_price"],
        alpha=0.25,
        s=18,
        label="Other listings",
    )
    ax.scatter(
        plot_data.loc[is_opportunity, "listed_price"],
        plot_data.loc[is_opportunity, "expected_price"],
        alpha=0.8,
        s=28,
        label=f"Potential deals (>{min_discount_pct}%)",
    )

    max_price = np.nanpercentile(
        np.r_[plot_data["listed_price"].values, plot_data["expected_price"].values],
        99,
    )
    ax.plot([0, max_price], [0, max_price], linestyle="--", color="black", linewidth=1)

    ax.set_xlim(0, max_price)
    ax.set_ylim(0, max_price)
    ax.set_xlabel("Precio publicado")
    ax.set_ylabel("Precio esperado por el modelo")
    ax.set_title("Precio esperado vs precio publicado")
    ax.grid(alpha=0.25)
    ax.legend()

    plt.tight_layout()
    plt.show()


plot_expected_vs_listed(deal_table, min_discount_pct=20)


<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
9. Interpretation and caveats
</h3>

Este análisis debe interpretarse como un **sistema de recomendación preliminar**, no como una garantía de ganancia. Que un auto aparezca subvaluado puede deberse a una oportunidad real, pero también a factores que el dataset no captura: estado mecánico, choques no informados, ubicación, urgencia de venta, deuda, equipamiento específico o errores de carga.

Para convertir esto en una aplicación más sólida, se podrían agregar filtros manuales:

- excluir autos con kilometraje extremo;
- excluir autos con errores evidentes de año o precio;
- exigir un mínimo de observaciones por marca-modelo;
- comparar cada oportunidad contra autos similares de la misma marca, modelo y año;
- revisar los top casos manualmente antes de sacar conclusiones.

Aun así, como extensión del trabajo, el enfoque es interesante porque transforma el modelo predictivo en una herramienta concreta para detectar posibles oportunidades de compra.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
10. TODOs for the final version
</h3>

Cosas que podemos completar después:

- Probar distintos umbrales (`10%`, `15%`, `20%`, `25%`) y justificar uno.
- Agregar una tabla final con el auto más subvaluado del dataset.
- Revisar manualmente los primeros 10 resultados para descartar errores evidentes.
- Comparar el ranking usando `discount_vs_expected_pct` versus `resale_gain_pct`.
- Guardar `deal_table` como CSV si queremos usarlo en una demo simple.
- Armar una función que reciba marca/modelo desde input de usuario.

<h3 style="background-color: #343a40; color: #ffffff; display: inline-block; padding: 6px 10px; border-radius: 6px;">
Posibles extensiones futuras
</h3>

Además del detector de autos subvaluados, se podrían desarrollar otras extensiones que agreguen valor al análisis y ayuden a interpretar mejor el mercado de SUVs usados. Algunas ideas interesantes son:

1. **Ranking de retención de valor por marca y modelo**  
   Analizar qué modelos pierden menos valor con el paso de los años. Para esto se podría comparar el precio mediano por año de fabricación dentro de cada `Marca` y `Modelo`, controlando por kilometraje cuando sea posible. Esta extensión sería útil porque conecta el modelo con una pregunta real de compra: qué autos conviene comprar si se busca minimizar la depreciación.

2. **Índice de oportunidad ajustado por riesgo**  
   El ranking actual identifica autos cuyo precio publicado está por debajo del esperado, pero no todos los casos tienen el mismo nivel de confianza. Se podría construir un indicador que combine descuento estimado, cantidad de autos similares disponibles y dispersión histórica de precios. Así, una oportunidad sería más confiable si pertenece a un grupo con muchas observaciones y precios relativamente estables.

3. **Comparación entre vendedor particular, concesionaria y tienda**  
   Evaluar si existe una diferencia sistemática de precio según `Tipo de vendedor`. Para hacerlo de forma más justa, no alcanza con comparar promedios: habría que controlar por marca, modelo, año y kilometraje. Si se encuentra una diferencia consistente, podría interpretarse como una prima de concesionaria o una posible ventaja de comprar a particulares.

4. **Simulador de precio esperado para una publicación nueva**  
   A partir del modelo entrenado, se podría armar una función donde el usuario ingrese características de un vehículo, como marca, modelo, año, kilómetros, transmisión y tipo de vendedor, y el sistema devuelva un precio estimado. Esta sería una aplicación práctica directa del modelo y permitiría evaluar si una publicación nueva parece cara, razonable o barata.

5. **Análisis de sensibilidad del precio**  
   Estimar cómo cambia el precio esperado al modificar una variable manteniendo el resto constante. Por ejemplo, cuánto baja el precio esperado al aumentar 10.000 km, o cuánto cambia al pasar de transmisión manual a automática. Esta extensión ayudaría a interpretar el modelo y no solo a medir su performance.

6. **Segmentación del mercado**  
   Separar los vehículos en grupos más homogéneos, por ejemplo gama alta, gama media y gama baja, o por rango de antigüedad. Luego se podría evaluar si los errores del modelo disminuyen dentro de cada segmento. Esta idea es útil porque el mercado no parece comportarse igual para todos los autos: un mismo error absoluto puede ser muy grande para un auto barato y menos relevante para uno de alta gama.

De estas alternativas, las más fuertes para sumar al trabajo serían el **ranking de retención de valor** y el **índice de oportunidad ajustado por riesgo**. La primera aporta interpretación económica del mercado, mientras que la segunda mejora la extensión actual y la vuelve más cuidadosa desde el punto de vista predictivo.
